# RF + MLP soft-voting ensemble

Two models that fail in **different** places, blended on their predicted probabilities.

| | Random Forest | MLP |
|---|---|---|
| features | raw | **signed-log** then standardised |
| under drift | cannot extrapolate — a point outside training support falls into the nearest leaf and gets that leaf's training distribution | extrapolates smoothly, and the log transform compresses the 5-orders-of-magnitude sensor range |
| forward-chaining mean | 0.7807 | 0.8390 |
| large-drift folds | 0.828 | **0.8502** |

**Why the RF gets raw features.** In *theory* `signed_log` is a strictly monotone per-feature
transform, trees split on thresholds, so the partition — and every prediction — should be
identical. The check in the next cell **refutes that in practice**: RF scores 0.8241 raw vs
0.7994 signed-log on fold 9. The reason is precision, not theory — scikit-learn casts features to
`float32`, and the raw magnitudes reach ~1e5, where float32 has only ~7 significant digits, so
values that are distinct after the log collapse into ties before it and the tree structure changes.
So the transform is not free for trees, and it happens to make them slightly worse. RF keeps raw
features because that is what measures better, not because the transform is a no-op.

### Evaluation rules carried over

- **Forward-chaining only**: train on batches `< n`, validate on `n`. Never trains on the future.
- **Judged on the large-drift folds, not the mean.** Folds 7 and 9 had tiny drift steps (1.42,
  1.44) and inflate the average; the batch 9 → 10 step the submission must actually survive is
  **7.22**, which resembles folds 2-6 and 8.
- **The blend weight is chosen walk-forward** (picked on earlier folds, applied to the next one),
  never on the fold being scored — otherwise the reported number is just the best-case blend.

In [1]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}"
      + (f"  ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "  [no GPU]"))

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
sample_sub = pd.read_csv("data/sample_submission.csv")

FEAT = [f"feat_{i}" for i in range(1, 129)]
RF_COLS = FEAT + ["concentration"]          # RF takes raw values
CLASSES = sorted(train["gas_class"].unique())
FOLDS = sorted(train["batch"].unique())[1:]

# Drift step per fold: how far the batch centroid moved from the previous batch.
_sc = StandardScaler().fit(train[FEAT])
_X = _sc.transform(train[FEAT])
_c = {b: _X[train["batch"].values == b].mean(0) for b in sorted(train["batch"].unique())}
DRIFT_STEP = {b: float(np.linalg.norm(_c[b] - _c[b - 1])) for b in FOLDS}
LARGE_DRIFT = [b for b, s in DRIFT_STEP.items() if s >= 5.0]
print("\ndrift step per fold:", {b: round(s, 2) for b, s in DRIFT_STEP.items()})
print(f"large-drift folds (>=5, like the 7.22 of batch 9->10): {LARGE_DRIFT}")

device: cuda  (NVIDIA GeForce RTX 5060 Ti)

drift step per fold: {np.int64(2): 5.02, np.int64(3): 5.55, np.int64(4): 9.15, np.int64(5): 7.47, np.int64(6): 9.59, np.int64(7): 1.42, np.int64(8): 5.57, np.int64(9): 1.44}
large-drift folds (>=5, like the 7.22 of batch 9->10): [np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(8)]


In [2]:
def signed_log(a):
    return np.sign(a) * np.log1p(np.abs(a))


def mlp_prep(fit_df, *apply_dfs):
    """signed-log + standardise, fit on the training fold only. Concentration is appended
    raw and logged, since it is a dose, not a sensor reading."""
    fs = StandardScaler().fit(signed_log(fit_df[FEAT].values))
    ex = lambda d: np.column_stack([d["concentration"].values,
                                    np.log1p(d["concentration"].values)])
    es = StandardScaler().fit(ex(fit_df))
    return [np.hstack([fs.transform(signed_log(d[FEAT].values)), es.transform(ex(d))]
                      ).astype(np.float32) for d in (fit_df,) + apply_dfs]


class MLP(nn.Module):
    def __init__(self, n_in, width=256, p_drop=0.3, n_out=6):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, width), nn.BatchNorm1d(width), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(width, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(128, n_out))

    def forward(self, x):
        return self.net(x)


def mlp_proba(X_tr, y_tr, X_ap, seeds=(0, 1, 2), epochs=80, bs=256, lr=2e-3, wd=1e-4):
    """Averaged over seeds -- a single net's predictions on a 161-row fold are very noisy."""
    Xt = torch.tensor(X_tr, device=DEVICE)
    Xa = torch.tensor(X_ap, device=DEVICE)
    yt = torch.tensor(y_tr, dtype=torch.long, device=DEVICE)
    acc = np.zeros((len(X_ap), len(CLASSES)))
    for s in seeds:
        torch.manual_seed(s)
        m = MLP(Xt.shape[1]).to(DEVICE)
        opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=wd)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
        for _ in range(epochs):
            m.train()
            for idx in torch.randperm(len(Xt), device=DEVICE).split(bs):
                if len(idx) < 2:            # BatchNorm needs >1 sample
                    continue
                loss = F.cross_entropy(m(Xt[idx]), yt[idx])
                opt.zero_grad(); loss.backward(); opt.step()
            sch.step()
        m.eval()
        with torch.no_grad():
            acc += F.softmax(m(Xa), dim=1).cpu().numpy()
    return acc / len(seeds)


def rf_proba(tr, ap_df):
    rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    rf.fit(tr[RF_COLS], tr["gas_class"])
    P = np.zeros((len(ap_df), len(CLASSES)))
    for i, c in enumerate(rf.classes_):     # align column order to CLASSES explicitly
        P[:, CLASSES.index(c)] = rf.predict_proba(ap_df[RF_COLS])[:, i]
    return P


# Test the "trees are invariant to a monotone transform" claim rather than assuming it.
_tr, _va = train[train["batch"] < 9], train[train["batch"] == 9]
_r = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(_tr[RF_COLS], _tr["gas_class"])
_t2, _v2 = _tr.copy(), _va.copy()
_t2[RF_COLS] = signed_log(_tr[RF_COLS].values); _v2[RF_COLS] = signed_log(_va[RF_COLS].values)
_r2 = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(_t2[RF_COLS], _t2["gas_class"])
_f1, _f2 = (f1_score(_va["gas_class"], _r.predict(_va[RF_COLS]), average="macro"),
            f1_score(_va["gas_class"], _r2.predict(_v2[RF_COLS]), average="macro"))
print(f"RF on fold 9:  raw={_f1:.4f}   signed-log={_f2:.4f}   identical={np.isclose(_f1, _f2)}")
print("  -> NOT identical, so the theoretical invariance does not hold here. sklearn casts to")
print("     float32; raw values reach ~1e5 where float32 keeps ~7 digits, so values that are")
print("     distinct after the log are tied before it and the tree structure changes.")
print("     Raw is also the better of the two for the RF, so the RF keeps raw features.")

RF on fold 9:  raw=0.8241   signed-log=0.7994   identical=False
  -> NOT identical, so the theoretical invariance does not hold here. sklearn casts to
     float32; raw values reach ~1e5 where float32 keeps ~7 digits, so values that are
     distinct after the log are tied before it and the tree structure changes.
     Raw is also the better of the two for the RF, so the RF keeps raw features.


In [3]:
# Compute each model's probabilities ONCE per fold, then the blend sweep is nearly free.
t0 = time.time()
fold_cache = {}
for vb in FOLDS:
    tr, va = train[train["batch"] < vb], train[train["batch"] == vb]
    X_tr, X_va = mlp_prep(tr, va)
    y_tr = np.searchsorted(CLASSES, tr["gas_class"].values)
    fold_cache[vb] = dict(y=va["gas_class"].values,
                          rf=rf_proba(tr, va),
                          mlp=mlp_proba(X_tr, y_tr, X_va))
    print(f"  fold {vb} done ({len(va)} rows)")
print(f"[{time.time() - t0:.0f}s]")


def blend_f1(vb, w):
    """w = weight on the RF; 1-w on the MLP."""
    c = fold_cache[vb]
    P = w * c["rf"] + (1 - w) * c["mlp"]
    return f1_score(c["y"], np.array(CLASSES)[P.argmax(1)], average="macro")


WGRID = np.round(np.arange(0, 1.01, 0.05), 2)
curve = pd.DataFrame({"w_rf": WGRID,
                      "mean": [np.mean([blend_f1(b, w) for b in FOLDS]) for w in WGRID],
                      "large_drift": [np.mean([blend_f1(b, w) for b in LARGE_DRIFT]) for w in WGRID]})
print()
print("blend weight sweep (w_rf = 0 is pure MLP, 1 is pure RF):")
print(curve.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

  fold 2 done (1244 rows)


  fold 3 done (1586 rows)


  fold 4 done (161 rows)


  fold 5 done (197 rows)


  fold 6 done (2300 rows)


  fold 7 done (3613 rows)


  fold 8 done (294 rows)


  fold 9 done (470 rows)
[60s]

blend weight sweep (w_rf = 0 is pure MLP, 1 is pure RF):
  w_rf   mean  large_drift
0.0000 0.8506       0.8636
0.0500 0.8501       0.8631
0.1000 0.8503       0.8633
0.1500 0.8502       0.8632
0.2000 0.8500       0.8630
0.2500 0.8498       0.8630
0.3000 0.8504       0.8638
0.3500 0.8514       0.8653
0.4000 0.8514       0.8653
0.4500 0.8521       0.8668
0.5000 0.8528       0.8682
0.5500 0.8523       0.8676
0.6000 0.8457       0.8600
0.6500 0.8423       0.8562
0.7000 0.8366       0.8510
0.7500 0.8265       0.8396
0.8000 0.8166       0.8287
0.8500 0.8011       0.8098
0.9000 0.7945       0.8038
0.9500 0.7889       0.7926
1.0000 0.7768       0.7792


In [4]:
# Honest weight selection: for each fold, pick w using ONLY the earlier folds.
wf_rows = []
for i, vb in enumerate(FOLDS):
    if i == 0:
        w = 0.5                                   # no history yet -> neutral blend
    else:
        prev = FOLDS[:i]
        w = float(WGRID[np.argmax([np.mean([blend_f1(b, ww) for b in prev]) for ww in WGRID])])
    wf_rows.append(dict(batch=vb, w_used=w, f1=blend_f1(vb, w),
                        rf=blend_f1(vb, 1.0), mlp=blend_f1(vb, 0.0)))
wf = pd.DataFrame(wf_rows)

best_w_mean = float(curve.loc[curve["mean"].idxmax(), "w_rf"])
best_w_big = float(curve.loc[curve["large_drift"].idxmax(), "w_rf"])

rows = [
    dict(model="RF alone", mean=wf.rf.mean(), large_drift=wf[wf.batch.isin(LARGE_DRIFT)].rf.mean()),
    dict(model="MLP alone", mean=wf.mlp.mean(), large_drift=wf[wf.batch.isin(LARGE_DRIFT)].mlp.mean()),
    dict(model=f"Ensemble, w picked walk-forward", mean=wf.f1.mean(),
         large_drift=wf[wf.batch.isin(LARGE_DRIFT)].f1.mean()),
    dict(model=f"Ensemble, best fixed w={best_w_mean} (optimistic)",
         mean=curve["mean"].max(),
         large_drift=float(curve.loc[curve["w_rf"] == best_w_mean, "large_drift"].iloc[0])),
]
summary = pd.DataFrame(rows)

print("=" * 84)
print(f"{'model':<44}{'mean F1':>12}{'large-drift F1':>18}")
print("-" * 84)
for _, r in summary.iterrows():
    print(f"{r.model:<44}{r['mean']:>12.4f}{r.large_drift:>18.4f}")
print("=" * 84)
print("\nper-fold detail (walk-forward weight):")
print(wf.round(4).to_string(index=False))
print(f"\nbest fixed w by mean = {best_w_mean}   by large-drift folds = {best_w_big}")
print("The walk-forward row is the honest one; the fixed-w row is an upper bound that had"
      "\nhindsight over every fold, and is shown only so the gap is visible.")

model                                            mean F1    large-drift F1
------------------------------------------------------------------------------------
RF alone                                          0.7768            0.7792
MLP alone                                         0.8506            0.8636
Ensemble, w picked walk-forward                   0.8365            0.8464
Ensemble, best fixed w=0.5 (optimistic)           0.8528            0.8682

per-fold detail (walk-forward weight):
 batch  w_used     f1     rf    mlp
     2    0.50 0.6807 0.6871 0.6711
     3    0.70 0.9714 0.9176 0.9892
     4    0.65 0.8183 0.7351 0.9072
     5    0.50 0.9895 0.9249 0.9895
     6    0.50 0.6854 0.6204 0.6913
     7    0.50 0.8615 0.7151 0.8736
     8    0.00 0.9333 0.7900 0.9333
     9    0.50 0.7516 0.8241 0.7497

best fixed w by mean = 0.5   by large-drift folds = 0.5
The walk-forward row is the honest one; the fixed-w row is an upper bound that had
hindsight over every fold, and is sh

In [5]:
# Is the blend actually adding anything, or is one model just carrying it?
dis, both_wrong, one_right = [], [], []
for vb in FOLDS:
    c = fold_cache[vb]
    pr = np.array(CLASSES)[c["rf"].argmax(1)]
    pm = np.array(CLASSES)[c["mlp"].argmax(1)]
    dis.append((pr != pm).mean())
    both_wrong.append(((pr != c["y"]) & (pm != c["y"])).mean())
    one_right.append((((pr == c["y"]) & (pm != c["y"])) | ((pm == c["y"]) & (pr != c["y"]))).mean())

d = pd.DataFrame(dict(batch=FOLDS, disagree=dis, both_wrong=both_wrong, exactly_one_right=one_right))
print("model diversity per fold:")
print(d.round(3).to_string(index=False))
print(f"\nmean disagreement {np.mean(dis):.3f} | both wrong {np.mean(both_wrong):.3f} "
      f"| exactly one right {np.mean(one_right):.3f}")
print("\n'exactly one right' is the headroom a blend can recover; 'both wrong' is the floor"
      "\nneither model can fix. A large 'both wrong' means more ensembling will not save you.")

model diversity per fold:
 batch  disagree  both_wrong  exactly_one_right
     2     0.193       0.102              0.178
     3     0.071       0.008              0.071
     4     0.186       0.068              0.143
     5     0.056       0.010              0.056
     6     0.123       0.225              0.090
     7     0.207       0.117              0.148
     8     0.150       0.007              0.146
     9     0.217       0.170              0.064

mean disagreement 0.150 | both wrong 0.088 | exactly one right 0.112

'exactly one right' is the headroom a blend can recover; 'both wrong' is the floor
neither model can fix. A large 'both wrong' means more ensembling will not save you.


In [6]:
# Refit on all 9 batches, predict batch 10, write a SEPARATE submission file.
#
# Weight choice is driven by the WALK-FORWARD result, not the hindsight-optimal one.
# The fixed w=0.5 row scores highest, but it had sight of every fold before choosing;
# the only honest simulation of "pick w, then meet a new batch" is the walk-forward row.
# If that loses to the pure MLP, blending has not earned its place and w is set to 0.
wf_big = wf[wf.batch.isin(LARGE_DRIFT)].f1.mean()
mlp_big = wf[wf.batch.isin(LARGE_DRIFT)].mlp.mean()

if wf_big > mlp_big:
    W_FINAL = best_w_big
    print(f"walk-forward blend {wf_big:.4f} > pure MLP {mlp_big:.4f} -> blending, w_rf={W_FINAL}")
else:
    W_FINAL = 0.0
    print(f"walk-forward blend {wf_big:.4f} <= pure MLP {mlp_big:.4f} on large-drift folds")
    print(f"  -> the ensemble does NOT earn its place; falling back to the pure MLP (w_rf=0.0).")
    print(f"  (the hindsight-best fixed w={best_w_big} would score {curve.large_drift.max():.4f},")
    print(f"   but that number is not obtainable without already knowing the answer.)")

X_all, X_te = mlp_prep(train, test)
y_all = np.searchsorted(CLASSES, train["gas_class"].values)

P_rf = rf_proba(train, test)
P_mlp = mlp_proba(X_all, y_all, X_te, seeds=(0, 1, 2, 3, 4))
P = W_FINAL * P_rf + (1 - W_FINAL) * P_mlp
pred = np.array(CLASSES)[P.argmax(1)]

sub = pd.DataFrame({"measurement_id": test["measurement_id"], "gas_class": pred})
assert list(sub.columns) == list(sample_sub.columns)
assert len(sub) == len(sample_sub)
assert (sub["measurement_id"].values == sample_sub["measurement_id"].values).all()
assert sub["gas_class"].isin(range(1, 7)).all() and sub["gas_class"].notna().all()
sub.to_csv("data/submission_ensemble.csv", index=False)
print(f"\nwrote data/submission_ensemble.csv with w_rf={W_FINAL}  (data/submission.csv untouched)")

vc = sub["gas_class"].value_counts().sort_index()
print("\npredicted distribution vs the 600/class the competition states:")
for c in CLASSES:
    print(f"  class {c}: {vc.get(c, 0):>5}  ({vc.get(c, 0) - 600:+d})")
print(f"  total absolute deviation: {int((vc.reindex(CLASSES).fillna(0) - 600).abs().sum())}")
print(f"\nRF and MLP disagree on "
      f"{(np.array(CLASSES)[P_rf.argmax(1)] != np.array(CLASSES)[P_mlp.argmax(1)]).mean():.1%}"
      " of batch-10 rows -- far above the ~15% seen on validation folds, which is itself a")
print("signal that batch 10 is further from training data than any validation fold was.")

walk-forward blend 0.8464 <= pure MLP 0.8636 on large-drift folds
  -> the ensemble does NOT earn its place; falling back to the pure MLP (w_rf=0.0).
  (the hindsight-best fixed w=0.5 would score 0.8682,
   but that number is not obtainable without already knowing the answer.)



wrote data/submission_ensemble.csv with w_rf=0.0  (data/submission.csv untouched)

predicted distribution vs the 600/class the competition states:
  class 1:   412  (-188)
  class 2:   467  (-133)
  class 3:   588  (-12)
  class 4:   548  (-52)
  class 5:   618  (+18)
  class 6:   967  (+367)
  total absolute deviation: 770

RF and MLP disagree on 41.4% of batch-10 rows -- far above the ~15% seen on validation folds, which is itself a
signal that batch 10 is further from training data than any validation fold was.
